# Titanic EDA (Part A)

Exploratory analysis of the Seaborn Titanic sample: load/save fallback, missing-value policy, univariate outliers, bivariate survival rates, correlation, and interpreted charts.

## 1. Load and save fallback (`titanic.csv`)

`sns.load_dataset("titanic")` appears **only** in this notebook. The CSV is written so later pipeline steps can read local data without calling Seaborn again.

In [ ]:
import os
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

os.makedirs("figures", exist_ok=True)
sns.set_theme(style="whitegrid")

# Golden rule: load_dataset appears ONLY here in the EDA notebook
df = sns.load_dataset("titanic")
df.to_csv("titanic.csv", index=False)

print(df.shape)
df.info()
print(df.describe())

## 2. Missing values analysis

Exact missing percentages, then banded rules: drop rows if $<5\%$, impute if $5$–$30\%$, drop or recode the column if $>30\%$.

In [ ]:
miss = (df.isna().mean() * 100).round(2)
print(miss[miss > 0])

### Missing-value justifications

- **`embarked` / `embark_town` (~0.22% missing):** $<5\%$ band $\rightarrow$ drop rows (`df.dropna(subset=['embarked', 'embark_town'])`). Two rows cannot support a reliable category-specific fill, and dropping them does not change class/sex mix in a material way.
- **`age` (~19.87% missing):** $5$–$30\%$ band $\rightarrow$ impute with median (robust to outliers and right-skew). Mean would be pulled by older passengers; a model-based fill is overkill at this stage.
- **`deck` (~77.2% missing):** Far above the $30\%$ threshold $\rightarrow$ drop the column **or** map to a `"Missing"` category (missingness is informative, mostly recorded for upper classes). Here we map to `"Missing"` so the signal is kept for later modeling.

The next cell applies those rules before univariate plots so later tables match the documented policy.

In [ ]:
n_before = len(df)
age_median = df["age"].median()

df = df.dropna(subset=["embarked", "embark_town"])
df["age"] = df["age"].fillna(age_median)
df["deck"] = df["deck"].astype("object").fillna("Missing")

print(f"Dropped {n_before - len(df)} rows with missing embarked/embark_town")
print(f"Age imputed with median = {age_median}")
print("Deck value counts (including Missing):")
print(df["deck"].value_counts())
print("Remaining missing %:")
print((df.isna().mean() * 100).round(2)[(df.isna().mean() * 100) > 0])

## 3. Univariate analysis and outliers

Histograms and boxplots for `age` and `fare`, IQR outlier counts, and fare central-tendency (right-skew: mean $>$ median $>$ mode).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.histplot(df["age"], bins=30, kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Age histogram")
sns.boxplot(x=df["age"], ax=axes[0, 1])
axes[0, 1].set_title("Age boxplot")

sns.histplot(df["fare"], bins=40, kde=True, ax=axes[1, 0])
axes[1, 0].set_title("Fare histogram")
sns.boxplot(x=df["fare"], ax=axes[1, 1])
axes[1, 1].set_title("Fare boxplot")

plt.tight_layout()
plt.savefig("figures/age_fare_univariate.png", dpi=150, bbox_inches="tight")
plt.show()

for col in ["age", "fare"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    outliers = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    print(f"{col} IQR outliers count: {outliers}")

fare_mean = df["fare"].mean()
fare_median = df["fare"].median()
fare_mode = df["fare"].mode().iloc[0]
print(f"fare mean: {fare_mean:.4f}")
print(f"fare median: {fare_median:.4f}")
print(f"fare mode: {fare_mode:.4f}")
print("Right-skew check (mean > median > mode):", fare_mean > fare_median > fare_mode)

Age is roughly unimodal around the late 20s after median fill; the boxplot still shows a long upper tail (older passengers). Fare is strongly right-skewed: most tickets cluster at low prices while a few first-class fares stretch the IQR fence, so **mean $>$ median $>$ mode**. Those fare outliers are real luxury tickets, not data errors, so they stay in the table for bivariate work.

## 4. Bivariate analysis and correlation matrix

Survival rates via boolean masks for `sex`, `pclass`, and combined segments, then a $6\times 6$ heatmap on `["survived", "pclass", "age", "sibsp", "parch", "fare"]`.

In [ ]:
print("Survival rate by sex")
print("  female:", df.loc[df["sex"] == "female", "survived"].mean())
print("  male:  ", df.loc[df["sex"] == "male", "survived"].mean())

print("\nSurvival rate by pclass")
for p in sorted(df["pclass"].unique()):
    print(f"  pclass={p}:", df.loc[df["pclass"] == p, "survived"].mean())

print("\nSurvival rate by sex × pclass")
for sex in ["female", "male"]:
    for p in [1, 2, 3]:
        mask = (df["sex"] == sex) & (df["pclass"] == p)
        print(f"  {sex}, pclass={p}: n={mask.sum()}, rate={df.loc[mask, 'survived'].mean():.4f}")

features = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
corr = df[features].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Titanic Feature Correlation Heatmap")
plt.savefig("figures/correlation_heatmap.png")
plt.show()
print(corr.round(3))

Women survive at a much higher rate than men; first class beats third class; first-class women approach certainty of survival while third-class men are near the floor. The heatmap agrees: `survived` is negatively correlated with `pclass` (higher class number = worse odds) and positively with `fare`. `sibsp`/`parch` are weakly related to survival; `age` is only a mild negative correlate after median fill.

## 5. Data story and visualizations

Four charts with a short interpretation under each.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=df, x="sex", y="survived", errorbar=None, ax=ax, color="steelblue")
ax.set_ylabel("Survival rate")
ax.set_title("Survival rate by sex")
plt.tight_layout()
plt.savefig("figures/survival_by_sex.png", dpi=150, bbox_inches="tight")
plt.show()

**Survival by sex.** Female passengers survive at roughly three-quarters; male passengers are closer to one-fifth. The gap is too large to be sampling noise in 891 rows and matches the “women and children first” boarding policy. Sex is therefore the strongest single split in this table and should stay in any later model.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=df, x="pclass", y="survived", hue="sex", errorbar=None, ax=ax)
ax.set_ylabel("Survival rate")
ax.set_title("Survival rate by class and sex")
plt.tight_layout()
plt.savefig("figures/survival_by_class_sex.png", dpi=150, bbox_inches="tight")
plt.show()

**Survival by class and sex.** First- and second-class women are almost all rescued; third-class women still out-survive first-class men. Male survival falls sharply from first to third class. Class and sex interact: a single additive effect would miss how poorly third-class men fared relative to every other segment.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=df, x="age", hue="survived", bins=30, kde=True, ax=ax, palette="coolwarm")
ax.set_title("Age distribution by survival")
plt.tight_layout()
plt.savefig("figures/age_by_survival.png", dpi=150, bbox_inches="tight")
plt.show()

**Age by survival.** Children are over-represented among survivors relative to their share of the passenger list; the adult peak around the imputed median is mixed. Older adults do not show a large survival advantage. Age matters mainly at the young end, which is why a median fill for missing adult ages is acceptable for a first model even if it slightly spikes the center of the histogram.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df, x="pclass", y="fare", hue="survived", ax=ax)
ax.set_title("Fare by class and survival")
plt.tight_layout()
plt.savefig("figures/fare_by_class_survival.png", dpi=150, bbox_inches="tight")
plt.show()

**Fare by class and survival.** Fare tracks class: first-class tickets sit far above second and third, with a long right tail of luxury fares. Within a class, survivors tend to sit slightly higher on fare (better cabins / family groups on more expensive tickets), but the between-class gap dominates. Fare is therefore partly a proxy for `pclass`; keep both only if the extra within-class signal is useful.